In [76]:

import tensorflow as tf
to_categorical = tf.keras.utils.to_categorical
array_to_img = tf.keras.preprocessing.image.array_to_img
ReduceLROnPlateau = tf.keras.callbacks.ReduceLROnPlateau
ImageDataGenerator = tf.keras.preprocessing.image.ImageDataGenerator
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objs as go
import plotly.figure_factory as ff
import plotly.express as px
import numpy as np
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
import pandas as pd

In [77]:
img_rows, img_cols = 224 , 224
batch_size = 8
epochs = 100
num_classes = 2
seed = 1

input_shape = (img_rows, img_cols, 3)
devide_list = tf.test.gpu_device_name()

In [78]:
datagen = ImageDataGenerator(rescale = 1./255,
                             horizontal_flip = True,
                             rotation_range=0.5,
                             zoom_range = 0.2       
                             )

val_test_datagen = ImageDataGenerator(rescale = 1./255)

In [79]:
train_it = datagen.flow_from_directory("D:/Portfolio/Academic Projects (June 2020 - June 2024)/05_Year-3_Perfume_Authenticity_CNN/dataset/Versace/train",
                                       target_size=(img_rows, img_cols),
                                       batch_size=batch_size,
                                       class_mode='binary',
                                       shuffle=True,
                                       color_mode='rgb',
                                       seed=seed)
test_it = val_test_datagen.flow_from_directory("D:/Portfolio/Academic Projects (June 2020 - June 2024)/05_Year-3_Perfume_Authenticity_CNN/dataset/Versace/test",
                                               target_size=(img_rows, img_cols),
                                               batch_size=batch_size,
                                               class_mode='binary',
                                               shuffle=False,
                                               color_mode='rgb',
                                               seed=seed)

val_it = val_test_datagen.flow_from_directory("D:/Portfolio/Academic Projects (June 2020 - June 2024)/05_Year-3_Perfume_Authenticity_CNN/dataset/Versace/val",
                                              target_size=(img_rows, img_cols),
                                              batch_size=batch_size,
                                              class_mode='binary',
                                              shuffle=False,
                                              color_mode='rgb',
                                              seed=seed)

Found 352 images belonging to 2 classes.
Found 44 images belonging to 2 classes.
Found 44 images belonging to 2 classes.


In [80]:
batchX, batchY = train_it.next()
print('Batch shape=%s, min=%.3f, max=%.3f' % (batchX.shape, batchX.min(), batchX.max()))

Batch shape=(8, 224, 224, 3), min=0.004, max=1.000


In [81]:
labels = list(train_it.class_indices.keys())
print(labels)

['Fake', 'Real']


In [82]:
print(Counter(train_it.classes))

Counter({0: 176, 1: 176})


In [83]:
weights = compute_class_weight('balanced', classes =  np.unique(train_it.classes), y = train_it.classes)
print(weights)

[1. 1.]


In [84]:
cw = dict(zip(np.unique(train_it.classes), weights))
print(cw)

{0: 1.0, 1: 1.0}


In [85]:
#Feature Extraction
model = tf.keras.Sequential()
model.add(tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

model.add(tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

model.add(tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

model.add(tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

#Image Classification
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

In [86]:
STEP_SIZE_TRAIN=train_it.n//train_it.batch_size
STEP_SIZE_VALID=val_it.n//val_it.batch_size
STEP_SIZE_TEST=test_it.n//test_it.batch_size

In [87]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_16 (Conv2D)          (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d_16 (MaxPoolin  (None, 111, 111, 32)     0         
 g2D)                                                            
                                                                 
 conv2d_17 (Conv2D)          (None, 109, 109, 32)      9248      
                                                                 
 max_pooling2d_17 (MaxPoolin  (None, 54, 54, 32)       0         
 g2D)                                                            
                                                                 
 conv2d_18 (Conv2D)          (None, 52, 52, 64)        18496     
                                                                 
 max_pooling2d_18 (MaxPoolin  (None, 26, 26, 64)      

In [88]:
learning_rate_reduction = ReduceLROnPlateau(monitor='val_loss', patience = 2, verbose=1,factor=0.1, min_lr=0.000001)
callbacks_list = [learning_rate_reduction]

In [89]:
his = model.fit(
        train_it,
        validation_data=val_it,
        epochs=epochs,
        class_weight=cw,
        callbacks=callbacks_list,
        shuffle=True,
        steps_per_epoch=STEP_SIZE_TRAIN,
        validation_steps=STEP_SIZE_VALID)

Epoch 1/100
44/44 [==============================] - 14s 302ms/step - loss: 0.7264 - accuracy: 0.4574 - val_loss: 0.6949 - val_accuracy: 0.4500 - lr: 0.0010
Epoch 2/100
44/44 [==============================] - 13s 297ms/step - loss: 0.6957 - accuracy: 0.5085 - val_loss: 0.6929 - val_accuracy: 0.5000 - lr: 0.0010
Epoch 3/100
44/44 [==============================] - 13s 294ms/step - loss: 0.6844 - accuracy: 0.5511 - val_loss: 0.7762 - val_accuracy: 0.4750 - lr: 0.0010
Epoch 4/100
44/44 [==============================] - ETA: 0s - loss: 0.6811 - accuracy: 0.5597
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
44/44 [==============================] - 13s 295ms/step - loss: 0.6811 - accuracy: 0.5597 - val_loss: 0.7607 - val_accuracy: 0.4500 - lr: 0.0010
Epoch 5/100
44/44 [==============================] - 13s 296ms/step - loss: 0.6835 - accuracy: 0.5199 - val_loss: 0.7024 - val_accuracy: 0.4750 - lr: 1.0000e-04
Epoch 6/100
44/44 [==============================] 

In [100]:
h1 = go.Scatter(y=his.history['loss'], 
                    mode="lines",
                    line=dict(
                        width=2,
                        color='blue'),
                        name="loss"
                   )
h2 = go.Scatter(y=his.history['val_loss'], 
                    mode="lines",
                    line=dict(
                        width=2,
                        color='red'),
                        name="val_loss"
                   )
                   
data = [h1,h2]
layout1 = go.Layout(title='Loss',
                   xaxis=dict(title='epochs'),
                   yaxis=dict(title=''))
fig1 = go.Figure(data, layout=layout1)
plotly.offline.iplot(fig1)

In [101]:
h1 = go.Scatter(y=his.history['accuracy'], 
                    mode="lines", line=dict(
                    width=2,
                    color='blue'),
                    name="acc"
                   )
h2 = go.Scatter(y=his.history['val_accuracy'], 
                    mode="lines", line=dict(
                    width=2,
                    color='red'),
                    name="val_acc"
                   )

data = [h1,h2]
layout1 = go.Layout(title='Accuracy',
                   xaxis=dict(title='epochs'),
                   yaxis=dict(title=''))
fig1 = go.Figure(data = data, layout=layout1)
plotly.offline.iplot(fig1)

In [102]:
score = model.evaluate(test_it,verbose=0)
print("Test Loss:",score[0])
print("Test Accuracy:",score[1])

Test Loss: 0.525458037853241
Test Accuracy: 0.7954545617103577


In [103]:
predicted_classes = (model.predict(test_it, verbose=1) > 0.5).astype("int32")[:,0]

6/6 [==============================] - 0s 63ms/step


In [104]:
predicted_classes[:10]

array([1, 0, 0, 1, 1, 0, 0, 0, 0, 0])

In [105]:
test_it.classes[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [106]:
def cm_plot(cm, labels):
    x = labels
    y = labels

    z_text = [[str(y) for y in x] for x in cm]
    fig = ff.create_annotated_heatmap(cm, x=x, y=y, annotation_text=z_text, colorscale='blues')

    fig.update_layout(title_text='Confusion Matrix')

    fig.add_annotation(dict(font=dict(color="black",size=13),
                            x=0.5,
                            y=-0.15,
                            showarrow=False,
                            text="Predicted Value",
                            xref="paper",
                            yref="paper"
                           ))

    fig.add_annotation(dict(font=dict(color="black",size=13),
                            x=-0.20,
                            y=0.5,
                            showarrow=False,
                            text="Real Value",
                            textangle=-90,
                            xref="paper",
                            yref="paper"
                           ))

    fig.update_layout(margin=dict(t=50, l=200))
    fig['layout']['yaxis']['autorange'] = "reversed"

    fig['data'][0]['showscale'] = True
    fig.show()

In [107]:
Counter(test_it.classes)

Counter({0: 22, 1: 22})

In [108]:
cm = confusion_matrix(test_it.classes, predicted_classes)
cm

array([[17,  5],
       [ 4, 18]], dtype=int64)

In [99]:
cm_plot(cm, labels)